In [41]:
%pip install optuna
%pip install catboost

In [42]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, classification_report
from scipy.sparse import hstack

import optuna
from catboost import CatBoostClassifier

In [43]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'(\d)\s*(mg|ml|kg|mcg|iu|units|g|mm|cm|mmol|meq|%)\b', r'\1\2', text)
    text = re.sub(r'[^\w\s\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [44]:
def make_synthetic_labels(n=1200, seed=42):
    """
    Generates synthetic drug data with a TRUE condition category.
    The 'condition_category' column is the actual label used to generate the condition text.
    """
    rng = np.random.default_rng(seed)
    condition_kw = {
        'Infection': ['bacterial infection', 'pneumonia and sinusitis', 'severe sepsis'],
        'Cardiovascular': ['hypertension and arrhythmia', 'coronary artery disease', 'cardiovascular risk'],
        'Neurological': ['epilepsy and seizure', 'migraine and neuropathy', 'parkinson symptoms'],
        'Psychiatric': ['depression and anxiety', 'bipolar disorder', 'schizophrenia symptoms'],
        'Respiratory': ['asthma and copd', 'bronchitis symptoms', 'pulmonary congestion'],
        'Endocrine': ['type 2 diabetes', 'hypothyroid condition', 'adrenal insufficiency'],
        'Pain': ['chronic pain and arthritis', 'migraine headache', 'gout flare'],
        'Gastrointestinal': ['gastric ulcer', 'crohn disease', 'ibs symptoms'],
        'Dermatology': ['acne and psoriasis', 'eczema dermatitis', 'skin rash'],
        'Cancer': ['breast carcinoma', 'lymphoma treatment', 'tumor chemotherapy'],
        'Autoimmune': ['rheumatoid arthritis', 'lupus symptoms', 'immunosuppressant therapy'],
        'Renal': ['renal impairment', 'kidney dialysis support', 'nephrotic syndrome'],
        'Ophthalmology': ['ophthalmic glaucoma', 'ocular conjunctivitis', 'cataract related'],
        'ENT': ['ear nose throat infection', 'sinus pharyngitis', 'nasal congestion'],
    }
    pregnancy_txt = {
        'Dangerous': ['contraindicated in pregnancy, may cause fetal harm', 'teratogenic, should not be used during pregnancy'],
        'Caution': ['use only if potential benefit justifies risk to fetus', 'caution advised, limited data in pregnancy'],
        'Consult': ['consult a healthcare provider before use in pregnant women', 'ask physician for professional advice'],
        'Safe': ['no known risk in pregnancy category a', 'no evidence of adverse effect in pregnancy'],
        'Unknown': ['', 'no data available'],
    }
    side_effect_txt = {
        'Life-threatening': ['risk of anaphylaxis and cardiac arrest', 'fatal hemorrhagic stroke reported'],
        'Severe': ['severe hepatotoxicity and liver necrosis', 'severe renal failure risk'],
        'Moderate': ['moderate nausea vomiting and headache', 'dizziness and hypotension reported'],
        'Mild': ['mild dry mouth and constipation', 'mild rash and pruritus'],
        'Unknown': ['', 'not established'],
    }
    pediatric_txt = {
        'Not Recommended': ['not recommended below the age of 12', 'contraindicated in pediatric patients'],
        'Safe': ['safe and effective in pediatric patients', 'approved for children'],
        'Consult': ['use with caution, consult physician', 'monitor pediatric patients closely'],
        'Unknown': ['', 'no pediatric data'],
    }
    effectiveness_txt = {
        5: ['highly effective with dramatic improvement'],
        4: ['significant improvement and demonstrated efficacy'],
        3: ['moderately effective with some improvement'],
        2: ['limited benefit, minimal improvement'],
        1: ['ineffective, no better than placebo'],
    }
    routes = ['ORAL', 'TOPICAL', 'INTRAVENOUS', 'INTRAMUSCULAR', 'INHALATION', 'OPHTHALMIC', 'NASAL']
    dosage_forms = ['TABLET', 'CAPSULE', 'CREAM', 'INJECTION', 'SOLUTION', 'SPRAY']

    rows = []
    conditions = list(condition_kw.keys())
    for i in range(n):
        cond = rng.choice(conditions)
        preg = rng.choice(list(pregnancy_txt.keys()), p=[0.1, 0.15, 0.25, 0.2, 0.3])
        se = rng.choice(list(side_effect_txt.keys()), p=[0.05, 0.15, 0.35, 0.3, 0.15])
        ped = rng.choice(list(pediatric_txt.keys()), p=[0.15, 0.35, 0.25, 0.25])
        eff = rng.choice(list(effectiveness_txt.keys()), p=[0.15, 0.3, 0.3, 0.15, 0.1])
        rows.append({
            'brand_name': f'Brand{i}', 'generic_name': f'Generic{i}',
            'substance_name': f'Substance{i}', 'manufacturer': f'Manufacturer{i % 40}',
            'route': rng.choice(routes), 'dosage_form': rng.choice(dosage_forms),
            'product_type': 'HUMAN PRESCRIPTION DRUG',
            'condition': f'Indicated for {rng.choice(condition_kw[cond])} in adults.',
            'contraindications': rng.choice(['hypersensitivity or allergy to the active ingredient',
                                              'severe hepatic disease', 'severe renal disease',
                                              'known cardiac arrhythmia', 'none known']),
            'pregnancy_warning': rng.choice(pregnancy_txt[preg]),
            'warnings': rng.choice([
                'see full prescribing information for warnings and precautions',
                'monitor liver function tests periodically during treatment',
                'caution advised in patients with cardiovascular disease history',
                'discontinue if signs of severe allergic reaction develop',
                'periodic renal function monitoring recommended during therapy']),
            'side_effects': rng.choice(side_effect_txt[se]),
            'drug_interactions': rng.choice(['concomitant use with other cns depressants may increase risk',
                                              'no significant interaction identified']),
            'dosage': f'{rng.choice([5,10,25,50,100,250,500])}mg once daily',
            'pediatric_use': rng.choice(pediatric_txt[ped]),
            'geriatric_use': rng.choice(['use with caution in elderly patients', 'no dose adjustment required']),
            'effectiveness': rng.choice(effectiveness_txt[eff]),
            # NEW: store the true condition category
            'condition_category': cond
        })
    return pd.DataFrame(rows)

In [ ]:
FILE_ID = '16VNd7qm9MpSbtK8uRX1y1cQ0OTx75MWz'   # Replace with your file's Google Drive ID


df_raw = None

try:
    with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as tmp:
        gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', tmp.name, quiet=False)
        df_raw = pd.read_csv(tmp.name, engine='python', on_bad_lines='skip')
        print(f"Loaded from Google Drive (ID: {FILE_ID}), shape={df_raw.shape}")
except Exception as e:
    print(f"Could not load from Drive: {e}")

# If that fails, generate synthetic data...please dont :D 
if df_raw is None:
    df_raw = make_synthetic_labels(n=1200, seed=42)
    print(f"Generated SYNTHETIC dataset, shape={df_raw.shape}")

# 3) Keep only clinically meaningful columns that exist
df = df_raw.dropna(axis=1, how='all').copy()
named_cols = ['brand_name', 'generic_name', 'substance_name', 'manufacturer',
              'route', 'dosage_form', 'product_type', 'condition',
              'contraindications', 'pregnancy_warning', 'warnings',
              'side_effects', 'drug_interactions', 'dosage',
              'pediatric_use', 'geriatric_use', 'effectiveness',
              'condition_category']
df = df[[c for c in named_cols if c in df.columns]].copy()
print("Shape after keeping clinically meaningful columns:", df.shape)

'openfda_labels.csv' not found or malformed -> generated a schema-matched SYNTHETIC dataset  shape=(1200, 18)
Shape after keeping clinically meaningful columns: (1200, 18)


In [46]:
text_cols = ['condition', 'contraindications', 'pregnancy_warning', 'warnings',
             'side_effects', 'drug_interactions', 'dosage', 'pediatric_use',
             'geriatric_use', 'effectiveness', 'brand_name', 'generic_name', 'substance_name']
for col in text_cols:
    if col in df.columns:
        df[col + '_clean'] = df[col].apply(clean_text)
print("Text cleaning complete.")

# Target (true labels, no rule-based post-processing)
target_col = 'condition_category'
y = df[target_col]

# Drop classes with too few samples (optional)
value_counts = y.value_counts()
df = df[y.isin(value_counts[value_counts >= 5].index)]
y = df[target_col]

Text cleaning complete.


In [47]:
feature_text_cols = ['brand_name_clean', 'generic_name_clean', 'substance_name_clean',
                     'contraindications_clean', 'pregnancy_warning_clean',
                     'warnings_clean', 'side_effects_clean', 'drug_interactions_clean',
                     'dosage_clean', 'pediatric_use_clean', 'geriatric_use_clean',
                     'effectiveness_clean']
# Ensure all exist; fill missing
for col in feature_text_cols:
    if col not in df.columns:
        df[col] = ''
    df[col] = df[col].fillna('').astype(str)

df['all_text'] = df[feature_text_cols].agg(' '.join, axis=1)

# TF‑IDF
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english', sublinear_tf=True)
X_text = tfidf.fit_transform(df['all_text'])

# Optional feature selection (SelectKBest)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
selector = SelectKBest(mutual_info_classif, k=2000)
X_text_selected = selector.fit_transform(X_text, y)

# Metadata: route, dosage_form (one‑hot)
meta_cols = ['route', 'dosage_form']
df_meta = df[meta_cols].fillna('unknown')
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_meta = encoder.fit_transform(df_meta)

# Combine
X = hstack([X_text_selected, X_meta])

In [48]:
n_components = 200   # tune this
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_pca = svd.fit_transform(X)
print(f"Reduced to {n_components} components, explained variance ratio: {svd.explained_variance_ratio_.sum():.3f}")

# Scale (optional but recommended for some models)
scaler = StandardScaler()
X_final = scaler.fit_transform(X_pca)

Reduced to 200 components, explained variance ratio: 0.964


In [50]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_final, y, groups=df['generic_name']))
X_train, X_test = X_final[train_idx], X_final[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 960, Test size: 240


In [54]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count': trial.suggest_int('border_count', 64, 255)
    }
    model = CatBoostClassifier(
        **params,
        loss_function='MultiClass',
        eval_metric='TotalF1',
        random_seed=42,
        verbose=False,
        auto_class_weights='Balanced'
    )
    model.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        early_stopping_rounds=30,
        use_best_model=True
    )
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='macro')

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)
best_params = study.best_params
print("Best parameters:", best_params)

[I 2026-08-01 09:06:41,063] A new study created in memory with name: no-name-48c9d5b1-5768-4dbe-98c4-5465ac53d2ab
[I 2026-08-01 09:07:16,752] Trial 0 finished with value: 0.17341354184908628 and parameters: {'iterations': 377, 'learning_rate': 0.016565821292526015, 'depth': 5, 'l2_leaf_reg': 6.880454549030126, 'border_count': 214}. Best is trial 0 with value: 0.17341354184908628.
[I 2026-08-01 09:07:36,511] Trial 1 finished with value: 0.12484358988767319 and parameters: {'iterations': 959, 'learning_rate': 0.03941480112514411, 'depth': 7, 'l2_leaf_reg': 5.315828195230188, 'border_count': 128}. Best is trial 0 with value: 0.17341354184908628.
[I 2026-08-01 09:07:42,453] Trial 2 finished with value: 0.15404210320192133 and parameters: {'iterations': 766, 'learning_rate': 0.0479092018305383, 'depth': 5, 'l2_leaf_reg': 9.186151048842783, 'border_count': 78}. Best is trial 0 with value: 0.17341354184908628.
[I 2026-08-01 09:08:08,018] Trial 3 finished with value: 0.16025617812112336 and pa

Best parameters: {'iterations': 520, 'learning_rate': 0.010982338253554628, 'depth': 4, 'l2_leaf_reg': 1.2384061032688383, 'border_count': 246}


In [56]:
final_model = CatBoostClassifier(
    **best_params,
    loss_function='MultiClass',
    eval_metric='TotalF1',
    random_seed=42,
    auto_class_weights='Balanced'
)
final_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=30,
    use_best_model=True
)

y_pred = final_model.predict(X_test)

0:	learn: 0.1081358	test: 0.0705941	best: 0.0705941 (0)	total: 157ms	remaining: 1m 21s
1:	learn: 0.1729798	test: 0.1484812	best: 0.1484812 (1)	total: 312ms	remaining: 1m 20s
2:	learn: 0.2232448	test: 0.1748521	best: 0.1748521 (2)	total: 466ms	remaining: 1m 20s
3:	learn: 0.2602992	test: 0.1772137	best: 0.1772137 (3)	total: 615ms	remaining: 1m 19s
4:	learn: 0.2963604	test: 0.1770311	best: 0.1772137 (3)	total: 795ms	remaining: 1m 21s
5:	learn: 0.3105875	test: 0.1728040	best: 0.1772137 (3)	total: 1s	remaining: 1m 26s
6:	learn: 0.3208172	test: 0.1649443	best: 0.1772137 (3)	total: 1.32s	remaining: 1m 36s
7:	learn: 0.3357860	test: 0.1795946	best: 0.1795946 (7)	total: 1.61s	remaining: 1m 43s
8:	learn: 0.3705429	test: 0.1791040	best: 0.1795946 (7)	total: 1.91s	remaining: 1m 48s
9:	learn: 0.3887523	test: 0.1819150	best: 0.1819150 (9)	total: 2.19s	remaining: 1m 51s
10:	learn: 0.4148261	test: 0.1600135	best: 0.1819150 (9)	total: 2.5s	remaining: 1m 55s
11:	learn: 0.4151965	test: 0.1668564	best: 0.1

In [57]:
print("\n===== TEST SET PERFORMANCE =====")
print("Accuracy:         ", round(accuracy_score(y_test, y_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred), 4))
print("Macro F1:         ", round(f1_score(y_test, y_pred, average='macro'), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


===== TEST SET PERFORMANCE =====
Accuracy:          0.225
Balanced Accuracy: 0.2228
Macro F1:          0.2159

Classification Report:
                   precision    recall  f1-score   support

      Autoimmune       0.21      0.27      0.24        15
          Cancer       0.15      0.17      0.16        18
  Cardiovascular       0.15      0.12      0.13        17
     Dermatology       0.23      0.19      0.21        16
             ENT       0.28      0.40      0.33        20
       Endocrine       0.15      0.20      0.17        15
Gastrointestinal       0.00      0.00      0.00        12
       Infection       0.22      0.32      0.26        19
    Neurological       0.38      0.19      0.25        16
   Ophthalmology       0.27      0.35      0.31        17
            Pain       0.07      0.06      0.06        17
     Psychiatric       0.38      0.24      0.29        21
           Renal       0.40      0.46      0.43        13
     Respiratory       0.21      0.17      0.19    

In [60]:
import joblib

# Package everything needed for inference into a single dictionary
model_package = {
    'model': final_model,
    'tfidf': tfidf,
    'selector': selector,
    'svd': svd,
    'scaler': scaler,
    'encoder': encoder,
    'target_classes': final_model.classes_
}

joblib.dump(model_package, 'drug_classifier_package.pkl')
print("Model package successfully saved as 'drug_classifier_package.pkl'")

Model package successfully saved as 'drug_classifier_package.pkl'
